# Heaps (Priority Queues)

A priority queue is a data structure with a few user-facing functions:
* **get:** always removes and returns the smallest (or the largest) value in the collection.
* **add:** places a new value in the collection
* **peek:** previews what's the smallest (or the largest) value in the collection.
* **size:** reports the number of items in the queue.

For this note, we'll focus on priority queues that return the smallest value.

A *trivial* way to implement a priority queue is to store the data in an ``array`` and keep the array sorted in ascending order. This guarantees that `array[0]` is the smallest value. As we apply the `get` method, array elements must be shifted to the left so that we always remove from position `[0]`; this would require $\Theta (n)$ time. 

Alternatively, we can have an integer pointer to the next element to remove from the sorted array, moving from `[0]`, to `[1]`, etc. This is a $\mathcal O(1)$ operation. However sorting the ``array`` takes $\Theta(n\log n)$ time every time we add an element to it.

If we do not sort the ``array`` but instead we perform a linear scan to find its minimum value, we can obtain it in $\Theta (n)$ which is faster than sorting.


There is even a faster way to obtain the same results. If the values are stored as a **complete binary tree,** data can be reordered in $\Theta (\log n)$ ensuring that the smallest value is always at the root of the tree. 

> **Definition:** A binary tree in which every level (depth), except possibly the deepest, is completely filled. At the last level, all nodes must be as far left as possible.<br>
> 
> Paul E. Black, "[complete binary tree](https://xlinux.nist.gov/dads/HTML/completeBinaryTree.html)", in Dictionary of Algorithms and Data Structures, 2016

A complete binary tree that guarantees its root is the smallest value is called a *minimum heap.* When the root is the largest value, it's called a *maximum heap.*

We'll focus on the minimum heap for now.

A minimum heap is a complete binary tree where 

* **the value of each parent node is less than or equal to the values of both its children.**  This is called the *minimum heap property;*
* a new value is added as the last leaf node in the tree and if the minimum heap property is violated, it must be restored;
* the root is always the smallest value in the tree and the value to be removed and returned during a `get` operation. The last leaf of the heap is moved to the vacant root. If the minimum heap property is violated, it must be restored;
* a violation of the heap property is restored by successive swaps between pairs of offending nodes.

## Adding data to a minimum heap

Adding data to a minimum heap is shown below. 

* **(i):** Initial state of the heap with 4 elements. Each node in this heap complies with the minimum heap property.
* **(ii):** Suppose we want to add the value $0$ to the heap. Initially it is added as the last leaf node of the complete binary tree. Immediately we notice that this new node and its parent node violate the minimum heap property. 
* **(iii):** We begin restoring the minimum heap property by swapping the newly added node and its parent. The new node moves up a level in the tree, but the minimum heap property is still violated.
* **(iv):** We continue restoring the minimum heap property by swapping the offending nodes. The property is finally restored.

In the worst case scenario, a new node at the bottom of the tree will have to climb all the way to the top, to restore the minimum heap property. For a complete binary tree, this requires $\Theta (\log n)$ steps because that's the number of levels in the tree (see below).


![](https://raw.githubusercontent.com/lgreco/images/refs/heads/main/graphs/min-heap-demo-add.drawio.png)


## Removing data from a minimum heap

Data from a minimum heap are always removed from the root -- that's where the data structure places its smallest element. The figure below shows what happens after the removal of the smallest element.

* **(i):** Initial state of the heap with 5 elements. Each node in this heap complies with the minimum heap property.
* **(ii):** The smallest element (always the root) has been removed and the root is now vacant. We replace the content of the root with the value of the last node of the complete binary tree.
* **(iii):** Moving that value to the root may violate the minimum heap property. In this case we swap the offending node with the smaller of its two children.
* **(iv):** After this swap, there are no more violations of the minimum heap property.

In the worst case scenario, the last value of the tree that we moved to the root node may have to sink all the way to the bottom, to restore the minimum heap property. For a complete binary tree, this requires $\Theta (\log n)$ steps because that's the number of levels in the tree (see below).

![](https://raw.githubusercontent.com/lgreco/images/refs/heads/main/graphs/min-heap-demo-remove.drawio.png)

## Moving data around a tree

In this note we number the levels of a tree starting at 1: the root is at level 1, its children are at level 2, and so on. A complete binary tree with $L$ levels has anywhere between $2^{L-1}$ and $2^L-1$ nodes. Therefore we can write that $L\in\Theta(\log_2 n)$ and for practical purposes:

In [ ]:
def L(n:int) -> int:
    """Given the number of nodes in a complete binary tree, return the number of levels in the tree (the root is level 1) using log base 2."""
    if n <= 0:
        raise ValueError("Number of nodes must be a positive integer.")
    import math
    return math.floor(math.log2(n)) + 1

Going up and down a binary tree may be fast but also requires significant memory to store the node objects and the left/right pointers. Essentially we are trading off time complexity for space complexity. There is a way to achieve the speed efficiency of the complete binary tree without sacrificing space (memory). It involves a humble array. This is shown below.

![](https://raw.githubusercontent.com/lgreco/images/refs/heads/main/graphs/Complete%20Binary%20Tree.drawio.png)


The tree is traversed level-by-level, from left to right, and its values are added to an array. For a tree with $L$ levels we know that there will be at most $2^L-1$ values so, we can size the array accordingly. The parent-children relationships can be preserved with the simple arithmetic below, where $n$ is the actual number of nodes in the tree and therefore the size of the array.

\begin{align}
\text{left}_\text{child}(\text{parent}) & = 2\times\text{parent}+1, && \ \text{parent} < \frac{n-1}{2} \\
\text{right}_\text{child}(\text{parent}) & = \text{left}_\text{child}(\text{parent}) +1, && \ \text{parent} < \frac{n-2}{2} \\
\text{parent}(\text{child}) & = \left\lfloor \frac{\text{child}-1}{2} \right\rfloor, && \ \text{child} > 0
\end{align}

The floor operator $\lfloor\,\rfloor$ is equivalent to *float-to-integer truncation* that is available in most programming languages. In Python, the truncation is implemented with the function `int()` but for *positive numbers* we can also obtain the same result with integer division (`//`). The simple arithmetic above is straightforward to code.

In [ ]:
def left_child(parent: int) -> int:
    """Return the index of the left child of the given parent index."""
    return 2 * parent + 1


def right_child(parent: int) -> int:
    """Return the index of the right child of the given parent index."""
    return left_child(parent) + 1


def parent(child: int) -> int:
    """Return the index of the parent of the given child index."""
    return (child - 1) // 2

## Mapping tree operations to array indices

When a complete binary tree for a minimum or a maximum heap is mapped as an array, its root is always in index position `[0]`. The last leaf node of the tree is in `[n-1]`, where `n` is the number of values stored in the priority queue.

Using these two positions, we can sketch the two basic operations for the heap:

**get:**
  ```python
  value_to_return = array[0]
  array[0] = array.pop()  # remove last item, assign it to first
  # TODO: verify heap property from root down; restore as needed
  ```
  The sketch assumes the heap holds at least two values: with exactly one value, `array.pop()` empties the array and the assignment to `array[0]` fails. In that case, simply pop and return the single value.

**add:**
  ```python
  array.append(new_value)
  # TODO: verify heap property from last element and up; restore as needed
  ```

As a reminder, for the remainder of this notebook, we **focus on a minimum heap.**

### Verify the heap property from the last leaf up

Once we add a leaf node, the only possible violation we can expect is between that leaf node and its parent. Because this is a minimum heap, a parent must be less than or equal to both its children. In this instance, a violation occurs if

```python
array[parent(len(array) - 1)] > array[len(array) - 1 ] 
```

The syntax above is cumbersome and can be improved with a few simple variables:

```python
new_node_idx = len(array) - 1
parent_idx = parent(new_node_idx)
if array[new_node_idx] < array[parent_idx]:
    # a violation has occurred
```

To repair the violation we simply swap the offending nodes:
```python
    temp = array[new_node_idx]
    array[new_node_idx] = array[parent_idx]
    array[parent_idx] = temp
```

There is one more step to complete the repair: make sure that the new node, in its new position, does not cause a new violation. For that, we need to update its place:
```python
    new_node_idx = parent_idx
```

Finally, we need to wrap everything in a loop that stops the moment no violation is observed or the new node finally floats to the root of the tree (which, in array terms, is the first position at index `[0]`).

### Verify the heap property from the root down

After a `get` operation, the last leaf node has been moved to the root. The only possible violation we can expect is between that node and its children. Because this is a minimum heap, a parent must be less than or equal to both its children. In this instance, a violation occurs if

```python
array[0] > array[left_child(0)] or array[0] > array[right_child(0)]
```

This time we need to be a bit more careful than when we were moving up the tree. Going up, every node except the root has exactly one parent. Going down, a node may have two children, only a left child, or no children at all (remember that the last level of a complete binary tree fills from the left). A child exists only if its index is within the array, that is, less than `len(array)`.

As before, a few simple variables make the syntax easier to read:

```python
node_idx = 0
left_idx = left_child(node_idx)
right_idx = right_child(node_idx)
```

If the node violates the minimum heap property, which child should we swap it with? We must always swap with the **smaller** of the two children. If we swapped with the larger child, it would become the parent of the smaller child and we would create a new violation. This is what happens in step **(iii)** of the removal figure above: the value $5$ is swapped with $1$, not with $2$. Assuming the node has at least a left child, we find the smaller child with:

```python
smaller_idx = left_idx
if right_idx < len(array) and array[right_idx] < array[left_idx]:
    smaller_idx = right_idx
```

Now we only need to compare the node with its smaller child. If the node is not greater than its smaller child, it is not greater than either child.

```python
if array[node_idx] > array[smaller_idx]:
    # a violation has occurred
```

To repair the violation we simply swap the offending nodes:
```python
    temp = array[node_idx]
    array[node_idx] = array[smaller_idx]
    array[smaller_idx] = temp
```

There is one more step to complete the repair: make sure that the node, in its new position, does not cause a new violation with its new children. For that, we need to update its place:
```python
    node_idx = smaller_idx
```

Finally, we need to wrap everything in a loop that stops the moment no violation is observed or the node finally sinks to a leaf of the tree. In array terms, a node is a leaf when it has no left child, that is, when `left_child(node_idx) >= len(array)`.

Each swap moves the node down one level. A node that starts at the root (level 1) can sink at most to level $L$, so the repair takes at most $L-1$ swaps, which is $\Theta(\log n)$ in the worst case.